# **G2P Evaluation of Epitran on Wiktionary Phonemic Data**

This notebook tests a rule-based G2P system, [Epitran](https://github.com/dmort27/epitran)
(Mortensen, Dalmia & Littell, 2018), on the same held-out test split as `Wik_eval.ipynb`. It is the
rule-based candidate in `G2P_compare.ipynb`, next to the statistical WFST model of
`Wik_eval.ipynb` and the neural models of `ByT5_eval.ipynb` and `ByT5_finetune.ipynb`.

Epitran converts spelling to IPA with hand-written rules for each language. Its Tagalog mode,
`tgl-Latn`, needs no training data, so it is used as it is. That also means it can't be
cross-validated: like `filipino-byt5-g2p` in `ByT5_eval.ipynb`, it is scored on each of the 10 cross-validation folds of
`Wik_eval.ipynb` without retraining.

Run `Wik_eval.ipynb` first: it writes the test split (`dicts/cwik_test.tsv`) and the
cross-validation pool (`dicts/cwik_trainval.dict`). Epitran is installed with
`pip install -r requirements.txt`; it doesn't need Phonetisaurus or a GPU.

**The process at a glance**

1. **Load** the test split written by `Wik_eval.ipynb`.
2. **Load** Epitran's Tagalog mode.
3. **Predict** every test word and save the predictions to `output/epitran_output.txt`.
4. **Score** the predictions with the same PER and WER code as `Wik_eval.ipynb`.
5. **Score** each of the 10 cross-validation folds of `Wik_eval.ipynb`.
6. **Compare** the result with the WFST model (`G2P_compare.ipynb` tests the difference).
7. **Analyze** the errors.

In [1]:
import os
import random
import sys
import time
from importlib.metadata import version
from pathlib import Path

import epitran
import numpy as np
import pandas as pd

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, compute_per_wer, collect_errors

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
output_dir  = current_dir / "output"

## 1. Load the test split

`Wik_eval.ipynb` saves its test split to `dicts/cwik_test.tsv`, one word per line with its phones
separated by spaces. The reference (`ref`) is built exactly as that notebook builds it.

In [2]:
test_tsv = dicts_dir / "cwik_test.tsv"
if not test_tsv.is_file():
    raise FileNotFoundError(f"{test_tsv} not found. Run Wik_eval.ipynb first: it writes the test split.")

ref = {}
with open(test_tsv, encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        ref[word] = phones.split()

print(f"Unique test words: {len(ref)}")

Unique test words: 2171


## 2. Load Epitran

Epitran's output needs two notation fixes before it can be compared with the references:

- It writes the affricates /t͡ʃ/ and /d͡ʒ/ without the tie bar (`tʃ`, `dʒ`), which
  `tokenize_ipa()` would split into two phones. The references use the tie bar, so they become
  `t͡ʃ` and `d͡ʒ`, the same fix as in `ByT5_eval.ipynb`.
- It keeps the hyphens and apostrophes of words like `mag-aral` and `pitumpu't`. They aren't
  phones, so they are removed.

Otherwise it uses the same symbols as the references, including `ɾ` for the tap, `ɡ` and `ʔ`.

In [3]:
epi = epitran.Epitran("tgl-Latn")
print("Epitran", version("epitran"))


def to_phones(ipa):
    """Tokenize Epitran output like the Wik_eval references: tie-bar affricates, no punctuation."""
    ipa = ipa.replace("-", "").replace("'", "").replace("tʃ", "t͡ʃ").replace("dʒ", "d͡ʒ")
    return tokenize_ipa("".join(ipa.split()))


def read_output(path):
    """Read a word TAB phones file into {word: [phones]}."""
    preds = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                preds[parts[0]] = parts[1].split()
    return preds


for word in ["araw", "kamisa", "kato", "mag-aral", "kasya"]:
    print(f"  {word:<10} {epi.transliterate(word)!r:<14} -> {' '.join(to_phones(epi.transliterate(word)))}")

Epitran 1.35.2
  araw       'ʔaɾaw'        -> ʔ a ɾ a w
  kamisa     'kamisa'       -> k a m i s a
  kato       'kato'         -> k a t o
  mag-aral   'maɡ-aɾal'     -> m a ɡ a ɾ a l
  kasya      'kaʃa'         -> k a ʃ a


## 3. Predict the test words

Every test word goes through Epitran. The predictions are saved to `output/epitran_output.txt` in
the same `word TAB phones` format as the WFST predictions.

In [4]:
start = time.time()
raw = {w: epi.transliterate(w) for w in ref}
hyp = {w: to_phones(s) for w, s in raw.items()}
print(f"Predicted {len(hyp)} words in {time.time() - start:.1f} s")

output_dir.mkdir(exist_ok=True)
epitran_output = output_dir / "epitran_output.txt"
with open(epitran_output, "w", encoding="utf-8") as f:
    for word, phones in hyp.items():
        f.write(f"{word}\t{' '.join(phones)}\n")

for word in list(ref)[:5]:
    print(f"  {word:<14} phones:[{' '.join(hyp[word])}]  ref:[{' '.join(ref[word])}]")

Predicted 2171 words in 0.1 s
  dagundong      phones:[d a ɡ u n d o ŋ]  ref:[d a ɡ u n d o ŋ]
  demonyo        phones:[d e m o ɲ o]  ref:[d e m o n j o]
  kustomer       phones:[k u s t o m e ɾ]  ref:[k u s t o m e ɾ]
  aneksyonismo   phones:[ʔ a n e k ʃ o n i s m o]  ref:[ʔ a n e k s j o n i s m o]
  ingklitik      phones:[ʔ i ŋ k l i t i k]  ref:[ʔ i ŋ k l i t i k]


## 4. Score the predictions

PER and WER are computed with `compute_per_wer()`, the function `Wik_eval.ipynb` uses. An `assert`
checks that every test word got a prediction.

Epitran reaches a PER of **4.80%** and a WER of **28.79%**: 1,546 of the 2,171 test words are fully
correct. Every test word got a prediction.

In [5]:
metrics = compute_per_wer(ref, hyp)
assert metrics["n_words"] == len(ref), f"scored {metrics['n_words']} of {len(ref)} test words"

print("=== Epitran (tgl-Latn) on the Wik_eval held-out test split ===")
print(f"  PER : {metrics['per']:.4f}  ({metrics['per'] * 100:.2f}%)   <- primary metric")
print(f"  WER : {metrics['wer']:.4f}  ({metrics['wer'] * 100:.2f}%)")
print(f"  Words evaluated  : {metrics['n_words']}")
print(f"  Correct words    : {metrics['n_correct']}")
print(f"  Total ref phones : {metrics['n_ref_phones']}")
print(f"  Total edit dist  : {metrics['n_edit']}")

=== Epitran (tgl-Latn) on the Wik_eval held-out test split ===
  PER : 0.0480  (4.80%)   <- primary metric
  WER : 0.2879  (28.79%)
  Words evaluated  : 2171
  Correct words    : 1546
  Total ref phones : 17303
  Total edit dist  : 831


## 5. Per-fold evaluation

The same 10 folds as the cross-validation in section 5 of `Wik_eval.ipynb`, rebuilt from
`dicts/cwik_trainval.dict` with the same seed. Epitran has no training step, so each fold is only
scored. The fold PERs are saved to `output/epitran_cv.tsv` for `G2P_compare.ipynb`.

The mean PER across the folds is **5.23%** (± 0.25%), close to its test PER.

In [6]:
trainval = []
with open(dicts_dir / "cwik_trainval.dict", encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        trainval.append((word, phones.split()))

# Rebuild Wik_eval's folds: the pool in file order, shuffled with the same seed.
random.seed(42)
cv_entries = trainval[:]
random.shuffle(cv_entries)

N_FOLDS = 10
fold_size = len(cv_entries) // N_FOLDS
folds = [cv_entries[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += cv_entries[N_FOLDS * fold_size:]   # remainder into last fold
print("Fold sizes:", [len(f) for f in folds])

cv_hyp = {w: to_phones(epi.transliterate(w)) for w in dict.fromkeys(w for w, _ in cv_entries)}

fold_pers = []
for k, held_out in enumerate(folds):
    fold_ref = {w: p for w, p in held_out}
    m = compute_per_wer(fold_ref, cv_hyp)
    fold_pers.append(m["per"])
    print(f"  Fold {k + 1:2d}: PER {m['per'] * 100:5.2f}%  ({m['n_words']} words)")

fold_pers = np.array(fold_pers)
pd.DataFrame({"fold": range(1, N_FOLDS + 1), "per": fold_pers}).to_csv(
    output_dir / "epitran_cv.tsv", sep="\t", index=False)
print()
print(f"Per-fold PER: mean {fold_pers.mean() * 100:.2f}%  +/-  {fold_pers.std() * 100:.2f}%")

Fold sizes: [1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1962]


  Fold  1: PER  5.32%  (1953 words)
  Fold  2: PER  5.41%  (1953 words)


  Fold  3: PER  5.28%  (1953 words)
  Fold  4: PER  5.07%  (1953 words)


  Fold  5: PER  5.15%  (1953 words)
  Fold  6: PER  4.87%  (1953 words)


  Fold  7: PER  5.80%  (1953 words)
  Fold  8: PER  4.96%  (1953 words)


  Fold  9: PER  5.22%  (1953 words)
  Fold 10: PER  5.21%  (1962 words)



Per-fold PER: mean 5.23%  +/-  0.25%


## 6. Compare with the WFST model

The WFST predictions in `output/cwik_output.txt` are scored against the same reference.

The WFST model is far better: a PER of **0.81%** against **4.80%**, and a WER of **5.85%**
against **28.79%**. Epitran makes 831 phone edits to the WFST's 141, almost six times as many.

In [7]:
rows = [{"Model": "Epitran (tgl-Latn)", **metrics}]
cwik_output = output_dir / "cwik_output.txt"
if cwik_output.is_file():
    rows.insert(0, {"Model": "WFST (Wik_eval)", **compute_per_wer(ref, read_output(cwik_output))})

summary = pd.DataFrame(rows)
summary["PER (%)"] = (summary["per"] * 100).round(2)
summary["WER (%)"] = (summary["wer"] * 100).round(2)
summary[["Model", "PER (%)", "WER (%)", "n_words", "n_correct", "n_ref_phones", "n_edit"]]

,Model,PER (%),WER (%),n_words,n_correct,n_ref_phones,n_edit
0,WFST (Wik_eval),0.81,5.85,2171,2044,17303,141
1,Epitran (tgl-Latn),4.80,28.79,2171,1546,17303,831


## 7. Error analysis

The errors by type, the phones behind them and the most common substitution pairs, as in
`Wik_eval.ipynb`.

There are **831** edits: 157 substitutions, 495 deletions and 179 insertions. Most come from
three gaps in the rules rather than from hard cases:

- **The glottal stop.** Epitran writes `ʔ` only at the start of a word, so it misses all the others:
  **387** deletions, 223 inside words and 164 at the end. That is almost half of its errors.
- **Spelling conventions of loanwords.** It reads *sy* and *ny* as single sounds, `ʃ` and `ɲ`,
  where the references keep two (`s j`, `n j`): `j → ʃ` (78, e.g. `malisyoso`) and `j → ɲ` (24,
  e.g. `demonyo`), each with a deleted `s` or `n`. It also writes *ts* as `t s` where the
  references often have `t͡ʃ` (29).
- **Spelled glides.** It reads the *y* of *-iya* and the *w* of *-uwa* as glides, where the
  references write only the vowels (`kuwarto` as /kuaɾto/): **105** extra `j` and **35** extra
  `w`.

In [8]:
err = collect_errors(ref, hyp)
total = err["n_sub"] + err["n_del"] + err["n_ins"]
display(pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of Errors": f"{err['n_sub'] / total * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of Errors": f"{err['n_del'] / total * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of Errors": f"{err['n_ins'] / total * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total,        "% of Errors": "100.0%"},
]))

print("Most often deleted:", err["del_counter"].most_common(5))
print("Most often inserted:", err["ins_counter"].most_common(5))
print("\nMost common substitution pairs:")
for (r, h), c in err["sub_pair_counter"].most_common(5):
    examples = ", ".join(ex["word"] for ex in err["sub_pair_examples"].get((r, h), [])[:3])
    print(f"  {r} -> {h}: {c}   e.g. {examples}")

,Type,Count,% of Errors
0,Substitution,157,18.9%
1,Deletion,495,59.6%
2,Insertion,179,21.5%
3,TOTAL,831,100.0%


Most often deleted: [('ʔ', 387), ('s', 78), ('n', 25), ('w', 1), ('ɡ', 1)]
Most often inserted: [('j', 105), ('w', 35), ('t', 29), ('s', 3), ('p', 2)]

Most common substitution pairs:
  j -> ʃ: 78   e.g. aneksyonismo, malisyoso, oblasyon
  t͡ʃ -> s: 29   e.g. solo-antso, maghitso, remolatsa
  j -> ɲ: 24   e.g. demonyo, milenyo, kalanbanyaga
  j -> i: 14   e.g. alhebraika, proteina, deins
  ʃ -> h: 3   e.g. shunga, kash, push
